In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# --- Smoothing function ---
def smooth(y, window=21, poly=3):
    return savgol_filter(y, window_length=window, polyorder=poly)

# --- Load data ---
df = pd.read_csv('random-duplicate-noisy.csv').sort_values('x').reset_index(drop=True)
X = df[['x']].values
x_plot = np.linspace(-20, 20, 500).reshape(-1, 1)

# --- True curves ---
def true_linear(x):    return 2*x + 5
def true_poly(x):      return x**3 - 4*x**2 + x + 6
def true_challenge(x): return np.sin(x) * np.exp(x/5)

functions = {
    'f(x) linear':     ('f_linear',     true_linear,    '#378ADD'),
    'g(x) polynomial': ('g_polynomial', true_poly,      '#1D9E75'),
    'h(x) challenge':  ('h_challenge',  true_challenge, '#7F77DD'),
}

poly_degrees = {
    'f(x) linear':     1,
    'g(x) polynomial': 3,
    'h(x) challenge':  8,
}

model_names  = ['Poly/Linear regression', 'Random forest', 'Neural network']
model_colors = {
    'Poly/Linear regression': '#E24B4A',
    'Random forest':          '#BA7517',
    'Neural network':         '#534AB7',
}

# --- Smoothing windows per function ---
# linear needs almost no smoothing, challenge needs the most
smooth_windows = {
    'f(x) linear':     11,
    'g(x) polynomial': 21,
    'h(x) challenge':  31,
}

r2_scores   = pd.DataFrame(index=list(functions.keys()), columns=model_names, dtype=float)
rmse_scores = pd.DataFrame(index=list(functions.keys()), columns=model_names, dtype=float)

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('Model predictions vs true curve — smoothed', fontsize=13, fontweight='normal', y=1.01)

for row, (fn_label, (col_name, true_fn, true_color)) in enumerate(functions.items()):

    y = df[col_name].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    X_train_sc = x_scaler.fit_transform(X_train)
    X_test_sc  = x_scaler.transform(X_test)
    y_train_sc = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

    # --- Fit models ---
    m1 = make_pipeline(PolynomialFeatures(degree=poly_degrees[fn_label]), LinearRegression())
    m1.fit(X_train, y_train)

    m2 = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
    m2.fit(X_train, y_train)

    m3 = MLPRegressor(
        hidden_layer_sizes=(128, 128, 64), activation='tanh',
        max_iter=20000, random_state=42, learning_rate='adaptive',
        learning_rate_init=0.001, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=100, tol=1e-6,
    )
    m3.fit(X_train_sc, y_train_sc)

    # --- Test set predictions for scoring (no smoothing) ---
    preds_m1 = m1.predict(X_test)
    preds_m2 = m2.predict(X_test)
    preds_m3 = y_scaler.inverse_transform(
        m3.predict(X_test_sc).reshape(-1, 1)
    ).ravel()

    # --- Full range predictions for plotting (smoothed) ---
    window = smooth_windows[fn_label]
    plot_m1 = smooth(m1.predict(x_plot), window=window)
    plot_m2 = smooth(m2.predict(x_plot), window=window)
    plot_m3 = smooth(y_scaler.inverse_transform(
        m3.predict(x_scaler.transform(x_plot)).reshape(-1, 1)
    ).ravel(), window=window)

    plot_preds = {
        'Poly/Linear regression': plot_m1,
        'Random forest':          plot_m2,
        'Neural network':         plot_m3,
    }

    # --- Score on unsmoothed predictions ---
    for mname, preds in zip(model_names, [preds_m1, preds_m2, preds_m3]):
        r2_scores.loc[fn_label, mname]   = round(r2_score(y_test, preds), 4)
        rmse_scores.loc[fn_label, mname] = round(np.sqrt(mean_squared_error(y_test, preds)), 4)

    # --- Plot ---
    for col, (mname, pred) in enumerate(plot_preds.items()):
        ax = axes[row][col]

        ax.scatter(X, y, color='#888780', alpha=0.2, s=4, label='noisy data')

        ax.plot(x_plot, true_fn(x_plot.ravel()),
                color=true_color, linewidth=1.8,
                linestyle='--', label='true curve')

        ax.plot(x_plot, pred, color=model_colors[mname],
                linewidth=1.8, label=mname)

        if row == 0:
            ax.set_title(mname, fontsize=11, fontweight='normal')
        if col == 0:
            ax.set_ylabel(fn_label, fontsize=9)

        ax.set_xlabel('x', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.2)
        ax.spines[['top', 'right']].set_visible(False)

        y_true_vals = true_fn(x_plot.ravel())
        margin = (y_true_vals.max() - y_true_vals.min()) * 0.3
        ax.set_ylim(y_true_vals.min() - margin, y_true_vals.max() + margin)

        if row == 0 and col == 0:
            ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('model_predictions_smoothed.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Heatmaps ---
fig, axes_h = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Model performance — smoothed predictions', fontsize=13, fontweight='normal', y=1.02)

sns.heatmap(r2_scores.astype(float), ax=axes_h[0],
            annot=True, fmt='.4f', cmap='Greens',
            vmin=0, vmax=1, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'R²'})
axes_h[0].set_title('R² score  (higher is better)', fontsize=11, fontweight='normal')
axes_h[0].tick_params(axis='x', rotation=15, labelsize=9)
axes_h[0].tick_params(axis='y', rotation=0, labelsize=9)

sns.heatmap(rmse_scores.astype(float), ax=axes_h[1],
            annot=True, fmt='.4f', cmap='Reds_r',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'RMSE'})
axes_h[1].set_title('RMSE  (lower is better)', fontsize=11, fontweight='normal')
axes_h[1].tick_params(axis='x', rotation=15, labelsize=9)
axes_h[1].tick_params(axis='y', rotation=0, labelsize=9)

plt.tight_layout()
plt.savefig('heatmap_smoothed.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nR² scores:\n', r2_scores)
print('\nRMSE scores:\n', rmse_scores)